# Vyuha P14 - adaptive-attack robustness ("the attacker moves second")

A *static* per-mutator ASR understates the attack surface: a resourced attacker **adapts** - it searches over compositions of the known transforms and keeps the strongest evasion. This eval runs that adaptive attacker (the defensive `RedTeam.run_adaptive`, which only mutates KNOWN seed attacks - no novel weaponization) against an **ablation** of the RJD detector to attribute Vyuha's robustness to its two design choices:

- **L0 de-obfuscation** (`norm`) - undoes encoding / spacing / homoglyph obfuscation.
- **adversarial augmentation** (`aug`) - trains the detector on mutated attacks.

The honest, measured finding: **L0 alone drops *static* ASR to ~0 but an *adaptive* attacker still reaches ~1.00** - and it is the augmentation, not L0, that closes the adaptive gap, without raising benign false positives. CPU-only, no API key.

In [ ]:
import sys, os, glob, subprocess
REPO_URL = "https://github.com/g25ait2149/vyuha.git"
DEST = "/kaggle/working/vyuha_src"
if os.path.isdir(os.path.join(DEST, ".git")):
    subprocess.run(["git", "-C", DEST, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, DEST], check=False)
hits = glob.glob(DEST + "/**/vyuha/__init__.py", recursive=True)
root = os.path.dirname(os.path.dirname(hits[0])) if hits else DEST
sys.path.insert(0, root)
for m in [m for m in sys.modules if m == "vyuha" or m.startswith(("vyuha.", "eval"))]:
    del sys.modules[m]
print("vyuha repo at:", root)

## A. Offline demo (built-in corpus) - the four-cell ablation
Static vs adaptive ASR and benign FPR for each (L0, aug) config. Reproduces the headline pattern in seconds.

In [ ]:
from eval.adaptive_eval import adaptive_robustness_eval
adaptive_robustness_eval(verbose=True)

## B. Real corpus (in-the-wild jailbreaks + benign)
Runs the same ablation on the real training corpus used in P1-P5. The adaptive search is O(seeds x ~100 transforms x 4 configs), so we cap the number of attack **seeds** (raise `N_SEEDS` for more coverage / more runtime). Benign set is subsampled for the FPR estimate.

In [ ]:
from eval.datasets import assemble
train_df, _ = assemble(verbose=True)
N_SEEDS, N_BENIGN = 150, 500
attacks = train_df[train_df.label == 1]['text'].head(N_SEEDS).tolist()
benign  = train_df[train_df.label == 0]['text'].head(N_BENIGN).tolist()
print(f'seeds={len(attacks)}  benign={len(benign)}')

rep = adaptive_robustness_eval(attacks=attacks, benign=benign, verbose=True)
rep['headline']

## Interpretation
- **vanilla (L0 off, aug off)** - both static and adaptive ASR high: a bare TF-IDF detector is fully evaded. This is the honest weakness baseline.
- **L0 only** - static ASR ~0 but **adaptive ASR high**: normalization defeats single-shot obfuscation, but an adaptive attacker composes transforms (esp. semantic wrappers L0 can't strip) and still gets through. This is the *attacker-moves-second premium* - the number static evaluations hide.
- **RJD-v2 (L0 + aug)** - adaptive ASR collapses while **benign FPR stays ~0**: adversarial augmentation is the load-bearing piece for adaptive robustness. Report this pair (adaptive ASR + benign FPR) as the robustness claim, never the static per-mutator number alone.